# Splunk SOAR artifact pull

Pulls the `"Field Values"` artifacts for a container from the SOAR REST API.

The `ph-auth-token` is read from `SOAR1_TOKEN` in the repo-root `.env` file (git-ignored; see `../.env.example`).


In [ ]:
import os

from dotenv import find_dotenv, load_dotenv
import json

import pandas as pd
import requests

SERVER = "https://prod-soar.mdr.tekstream.com"
load_dotenv(find_dotenv(usecwd=True))
TOKEN = os.environ.get("SOAR1_TOKEN", "")  # from .env
CONTAINER_ID = 232644
ARTIFACT_NAME = "Field Values"

assert TOKEN, "set SOAR1_TOKEN in .env (see ../.env.example)"
HEADERS = {"ph-auth-token": TOKEN, "Content-Type": "application/json"}


In [ ]:
params = {
    "_filter_container": CONTAINER_ID,
    "_filter_name": json.dumps(ARTIFACT_NAME),  # SOAR expects the value in double quotes
    "page_size": 0,  # 0 = return all results
}

response = requests.get(
    f"{SERVER}/rest/artifact",
    headers=HEADERS,
    params=params,
    allow_redirects=False,
    timeout=60,
)
print(response.status_code, response.url)
response.raise_for_status()

result = response.json()
print(f"count={result.get('count')}  num_pages={result.get('num_pages')}")

In [ ]:
# Raw JSON of the first artifact
artifacts = result.get("data", [])
print(json.dumps(artifacts[0] if artifacts else result, indent=2)[:5000])

In [ ]:
# Artifact metadata as a table
df = pd.json_normalize(artifacts, max_level=0)
df.drop(columns=["cef"], errors="ignore").head()

In [ ]:
# CEF field values, one row per artifact
cef = pd.DataFrame([a.get("cef", {}) for a in artifacts])
cef.insert(0, "artifact_id", [a.get("id") for a in artifacts])
cef